In [1]:
!pip install -q \
    langchain==0.2.0 \
    langchain-core==0.2.0 \
    langchain-community==0.2.0 \
    langchain-google-genai==1.0.3 \
    faiss-cpu==1.7.4

ERROR: Could not find a version that satisfies the requirement faiss-cpu==1.7.4 (from versions: 1.12.0, 1.13.0, 1.13.1, 1.13.2, 1.14.2)

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for faiss-cpu==1.7.4


In [1]:
import os
from pathlib import Path
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableBranch
from dotenv import load_dotenv


print("="*60)
print(" INITIATING ADVANCED RAG ROUTING ARCHITECTURE ")
print("="*60)

C:\Users\lenovo\AppData\Local\Temp\ipykernel_784\39872389.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


 INITIATING ADVANCED RAG ROUTING ARCHITECTURE 


In [2]:
# Load .env (development) into env vars; production should set real env vars or use a secrets manager.
load_dotenv()  # reads .env if present

# Prefer explicit env var; fall back to a secrets file only if provided
api_key = os.getenv("GEMINI_API_KEY") 

if not api_key:
    secrets_path = Path(os.getenv("SECRETS_PATH", Path("secrets") / "api"))
    if secrets_path.exists():
        with secrets_path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line.startswith("GEMINI_API_KEY="):
                    api_key = line.split("=", 1)[1].strip().strip('"').strip("'")
                    break

if not api_key:
    raise ValueError("GEMINI_API_KEY not found in environment or secrets file. Set GEMINI_API_KEY or SECRETS_PATH.")

os.environ["GEMINI_API_KEY"] = api_key

print("API key loaded:", bool(api_key))



API key loaded: True


In [ ]:
# Read and sanitize DB_DIR environment value
db_env = os.getenv("DB_DIR", "").strip()
if db_env.startswith(("r\"", "r'", "R\"", "R'")):
    db_env = db_env[1:]
db_env = db_env.strip().strip('\'"')

if db_env:
    DB_DIR = Path(db_env)
else:
    app_base = Path(os.getenv("APPDATA") or os.getenv("LOCALAPPDATA") or Path.home())
    DB_DIR = app_base / "ai-foundations-lab" / "faiss_extract_once_db"

DB_DIR = DB_DIR.expanduser()

# Safe mkdir (this will fail only if DB_DIR contains invalid characters)
DB_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash", temperature=0.0)
embedder = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

# Load our FAISS Database (The Knowledge Base)
vectorstore = FAISS.load_local(
    DB_DIR, embedder, allow_dangerous_deserialization=True
)
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
